In [ ]:
# --- Xray Cloud: Release Test Report (Unique Latest Run per Test, Charts & Graphs) ---

import requests, getpass, os, json, pandas as pd, datetime
from collections import Counter
from datetime import datetime as dt
from IPython.display import display
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Segoe UI Emoji", "DejaVu Sans"]
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

# ============================================================
# Global Config
# ============================================================
XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"
CACHE_DIR = "xray_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Disable SSL verify warning (local only)
import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
VERIFY_SSL = False

# ============================================================
# Auth
# ============================================================
CID  = getpass.getpass("XRAY_CLIENT_ID: ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET: ")

def auth():
    r = requests.post(XRAY_AUTH_URL, json={"client_id": CID, "client_secret": CSEC}, timeout=30, verify=VERIFY_SSL)
    r.raise_for_status()
    return r.text.strip('"')

TOKEN = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def gql(query, variables=None):
    r = requests.post(XRAY_GRAPHQL_URL, json={"query": query, "variables": variables or {}}, headers=HEADERS, timeout=60, verify=VERIFY_SSL)
    r.raise_for_status()
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# ============================================================
# Queries
# ============================================================
Q_TESTPLANS = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestPlans(jql:$jql, limit:$limit, start:$start) {
    results { issueId jira(fields:["key","summary","status"]) }
  }
}
"""

Q_PLAN_EXECS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    testExecutions(limit:$limit, start:$start) {
      total start limit
      results { issueId }
    }
  }
}
"""

Q_PLAN_TESTS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    tests(limit:$limit, start:$start) {
      total start limit
      results { jira(fields:["key"]) }
    }
  }
}
"""

Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results {
      status { name }
      test { jira(fields:["key"]) }
      startedOn
      finishedOn
    }
  }
}
"""

# ============================================================
# Helpers
# ============================================================
def get_test_plan_status(plan_key):
    d = gql(Q_TESTPLANS, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    plans = d["getTestPlans"]["results"]
    if not plans:
        return "Unknown"
    status = plans[0]["jira"].get("status")
    if isinstance(status, dict):
        return status.get("name", "Unknown")
    return str(status).strip().title() if status else "Unknown"

def cache_path_for(plan_key):
    return os.path.join(CACHE_DIR, f"{plan_key}.parquet")

def save_to_cache(plan_key, df):
    path = cache_path_for(plan_key)
    df.to_parquet(path, index=False)
    print(f"Saved cache for {plan_key} → {path}")

def load_from_cache(plan_key):
    path = cache_path_for(plan_key)
    if os.path.exists(path):
        print(f"Using cached data for {plan_key}")
        return pd.read_parquet(path)
    return None

# ============================================================
# Fetch from Xray
# ============================================================
def fetch_release_data(plan_key):

    # 1. Resolve Test Plan → issueId
    plan_d = gql(Q_TESTPLANS, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    if not plan_d["getTestPlans"]["results"]:
        raise SystemExit(f"No Test Plan found for {plan_key}")
    plan_id = plan_d["getTestPlans"]["results"][0]["issueId"]
    plan_status = plan_d["getTestPlans"]["results"][0]["jira"]["status"].get("name", "Unknown")

    # 2. Planned tests
    planned = set()
    start = 0
    while True:
        d = gql(Q_PLAN_TESTS, {"planId": plan_id, "limit": 200, "start": start})
        block = d["getTestPlan"]["tests"]
        for t in block["results"]:
            k = (t.get("jira") or {}).get("key")
            if k:
                planned.add(k)
        start += block["limit"]
        if start >= block["total"]:
            break

    df_plan = pd.DataFrame([{"plan_key": plan_key, "plan_status": plan_status, "source": "PLANNED_SCOPE", "test_key": k} for k in planned])

    # 3. Linked executions
    exec_ids = []
    start = 0
    while True:
        d = gql(Q_PLAN_EXECS, {"planId": plan_id, "limit": 100, "start": start})
        te = d["getTestPlan"]["testExecutions"]
        exec_ids += [x["issueId"] for x in te["results"]]
        start += te["limit"]
        if start >= te["total"]:
            break

    # 4. Test runs
    all_runs = []
    if exec_ids:
        BATCH = 50
        for i in range(0, len(exec_ids), BATCH):
            batch = exec_ids[i:i + BATCH]
            start = 0
            limit = 100
            while True:
                data = gql(Q_RUNS, {"execIds": batch, "limit": limit, "start": start})
                runs = data["getTestRuns"]
                for tr in runs["results"]:
                    sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
                    test_key = (tr.get("test") or {}).get("jira", {}).get("key")
                    all_runs.append({
                        "plan_key": plan_key,
                        "plan_status": plan_status,
                        "source": "EXECUTION_RUN",
                        "test_key": test_key,
                        "status": sname,
                        "finishedOn": tr.get("finishedOn")
                    })
                start += runs["limit"]
                if start >= runs["total"]:
                    break

    df_exec = pd.DataFrame(all_runs)
    df = pd.concat([df_plan, df_exec], ignore_index=True)
    return df

# ============================================================
# Dynamic Cache 
# ============================================================

def fetch_or_load(plan_key):
    """Smart cache loader that refreshes if plan is not Done/Closed."""
    path = cache_path_for(plan_key)
    plan_status = get_test_plan_status(plan_key)
    print(f"Plan {plan_key} status: {plan_status}")

    # Normalize status (case-insensitive)
    plan_status_norm = (plan_status or "").strip().lower()

    # Define what counts as "finalized"
    done_statuses = {"done", "closed", "completed"}
    
    # If not finalized → always fetch fresh data
    if plan_status_norm not in done_statuses:
        print(f"Plan is active ({plan_status}) → fetching fresh data from Xray...")
        df = fetch_release_data(plan_key)
        save_to_cache(plan_key, df)
        return df

    # If finalized & cache exists → reuse cached data
    if os.path.exists(path):
        try:
            cached = pd.read_parquet(path)
            print(f"Using cached data for {plan_key} (status = {plan_status})")
            return cached
        except Exception as e:
            print(f"Cache load failed for {plan_key}: {e} → refetching...")
            df = fetch_release_data(plan_key)
            save_to_cache(plan_key, df)
            return df

    # If finalized but no cache yet → fetch once and cache
    print(f" No cache found for {plan_key} (status = {plan_status}) → fetching fresh data...")
    df = fetch_release_data(plan_key)
    save_to_cache(plan_key, df)
    return df

# ============================================================
# Compute Release Metrics
# ============================================================
def compute_release_metrics(df_all):
    """Compute release metrics ensuring alignment between planned and executed tests."""
    summary = []

    for rel in df_all["plan_key"].unique():
        subset = df_all[df_all["plan_key"] == rel]
        plan_status = subset["plan_status"].dropna().unique()[0] if "plan_status" in subset.columns else "Unknown"

        # --- Planned tests list ---
        planned_tests = subset[subset["source"] == "PLANNED_SCOPE"]["test_key"].unique().tolist()
        planned = len(planned_tests)

        # --- Execution runs (latest per test only for planned scope) ---
        exec_subset = subset[subset["source"] == "EXECUTION_RUN"].copy()
        exec_subset = exec_subset[exec_subset["test_key"].isin(planned_tests)]

        # select latest finished run per test
        if "finishedOn" in exec_subset.columns:
            exec_subset["finishedOn"] = pd.to_datetime(exec_subset["finishedOn"], errors="coerce")
            exec_subset = exec_subset.sort_values(by=["test_key", "finishedOn"], ascending=True)
            latest_runs = exec_subset.groupby("test_key").tail(1)
        else:
            latest_runs = exec_subset.drop_duplicates(subset=["test_key"], keep="last")

        # --- Status counts (unique latest only) ---
        status_list = ["TO DO", "PASSED", "FAILED", "BLOCKED", "EXECUTING", "DESCOPED"]
        counts = latest_runs["status"].value_counts().reindex(status_list, fill_value=0)
        todo, passed, failed, blocked, executing, descoped = (
            counts["TO DO"], counts["PASSED"], counts["FAILED"],
            counts["BLOCKED"], counts["EXECUTING"], counts["DESCOPED"]
        )

        # --- Actual Execution (raw effort of pass/fail) ---
        actual_execution = len(subset[(subset["source"] == "EXECUTION_RUN") &
                                      (subset["status"].isin(["PASSED", "FAILED"]))])

        # --- Add missing To Dos ---
        known_status_tests = set(latest_runs["test_key"])
        missing_tests = set(planned_tests) - known_status_tests
        todo += len(missing_tests)

        # --- Metrics ---
        executed_tests = passed + failed + blocked + executing
        pass_rate = round((passed / (passed + failed) * 100), 2) if (passed + failed) else 0
        exec_coverage = round((executed_tests / planned * 100), 2) if planned else 0
        completion = round(((passed + failed + descoped + blocked + executing) / planned * 100), 2) if planned else 0
        remaining = round(((planned - (passed + failed + descoped + blocked + executing)) / planned * 100), 2) if planned else 0

        summary.append({
            "Release": rel,
            "Status": plan_status,
            "Planned Tests": planned,
            "Actual Execution (Raw)": actual_execution,
            "Execution Coverage %": exec_coverage,
            "Pass %": pass_rate,
            "Completion %": completion,
            "Remaining %": remaining,
            "To Do": todo,
            "Passed": passed,
            "Failed": failed,
            "Blocked": blocked,
            "Executing": executing,
            "Descoped": descoped
        })

    cols = [
        "Release", "Status", "Planned Tests", "Actual Execution (Raw)",
        "Execution Coverage %", "Pass %", "Completion %", "Remaining %",
        "To Do", "Passed", "Failed", "Blocked", "Executing", "Descoped"
    ]
    return pd.DataFrame(summary)[cols]

# ============================================================
# Run for Releases
# ============================================================
release_keys = [
    "PROJ-1001",
    "PROJ-1002",
    "PROJ-1003",
]

all_dfs = [fetch_or_load(rk) for rk in release_keys]
df_all = pd.concat(all_dfs, ignore_index=True)

dashboard_df = compute_release_metrics(df_all)

def highlight_health(val, col):
    """Color thresholds for Pass %, Completion %, Remaining %"""
    if pd.isna(val):
        return ""
    if col == "Pass %":
        if val >= 95: return "background-color:#C8E6C9;"  # Green
        elif val >= 90: return "background-color:#FFF9C4;"  # Yellow
        else: return "background-color:#FFCDD2;"  # Red
    if col == "Completion %":
        if val >= 95: return "background-color:#DCEDC8;"
        elif val >= 90: return "background-color:#FFF9C4;"
        else: return "background-color:#FFCDD2;"
    if col == "Remaining %":
        if val <= 5: return "background-color:#C8E6C9;"
        elif val <= 10: return "background-color:#FFF9C4;"
        else: return "background-color:#FFCDD2;"
    return ""

styled_dashboard = (
    dashboard_df.style
        .format({
            "Execution Coverage %": "{:.2f}%",
            "Pass %": "{:.2f}%",
            "Completion %": "{:.2f}%",
            "Remaining %": "{:.2f}%",
        })
        .apply(lambda col: [highlight_health(v, col.name) for v in col]
               if col.name in ["Pass %", "Completion %", "Remaining %"] else [""] * len(col))
        .bar(subset=["Execution Coverage %"], color="#2196F3", vmin=0, vmax=100)
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "14px"), ("font-weight", "bold"), ("color", "#333")]},
            {"selector": "th", "props": [("background-color", "#f5f5f5"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ])
        .set_caption("Release Test Metrics Dashboard")
)

display(styled_dashboard)

# Save to cache and CSV
dashboard_df.to_csv(os.path.join(CACHE_DIR, "release_metrics_dashboard.csv"), index=False)
dashboard_df.to_parquet(os.path.join(CACHE_DIR, "release_metrics_dashboard.parquet"), index=False)
print("\n Dashboard saved in xray_cache/")


# ============================================================
# Dashboard Visualization
# ============================================================
plt.style.use("seaborn-v0_8-muted")

def add_value_labels(ax, spacing=3):
    for rect in ax.patches:
        height = rect.get_height()
        ax.annotate(f"{height:.1f}%",
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, spacing),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, color='black')

# --- Chart 1: Pass % ---
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(dashboard_df["Release"], dashboard_df["Pass %"], color="#4CAF50")
ax.set_title("Pass % by Release", fontsize=14, fontweight='bold')
ax.set_xlabel("Release")
ax.set_ylabel("Pass %")
ax.grid(axis='y', linestyle="--", alpha=0.7)
add_value_labels(ax)
plt.tight_layout()
plt.show()



